In [15]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from sklearn.metrics import roc_auc_score

In [16]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [17]:
def ttest_one_var(test_data, feature_col, target_col, equal_var=False):
    """
    Perform an independent t-test for one numeric column against a binary target.
    
    Args:
        test_data (pd.dataframe): Numeric data to test
        feature_col (column name of feature): Numeric feature to test
        target_col (column name of target): Binary target (must have exactly 2 groups)  
        equal_var (bool): If False (default), uses Welch's t-test (unequal variances)
    
    Returns:
        dict: t-statistic, p-value, group means, and effect size (Cohen's d)
    
    Example:
        results = ttest_one_var(df, 'LTV', 'TARGET', equal_var=False)
    """
    # Check for exactly 2 groups
    groups = [0, 1]
    
    # Split data into two groups
    group1 = test_data.loc[test_data[target_col] == groups[0], feature_col].dropna()
    group2 = test_data.loc[test_data[target_col] == groups[1], feature_col].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=equal_var)
    
    # Calculate effect size (Cohen's d)
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1+n2-2))
    cohens_d = (group1.mean() - group2.mean()) / pooled_std
    
    return {
        'feature': feature_col,
        't_statistic': t_stat,
        'p_value': p_value,
        'mean_group1': group1.mean(),
        'mean_group2': group2.mean(),
        'cohens_d': cohens_d,
        'abs_t_stat': abs(t_stat)  # Useful for ranking variables
    }

In [18]:
def calculate_auc_one_var(df, numeric_col, target_col):
    """
    Simplified AUC calculation using sklearn's roc_auc_score.
    Returns json with AUC scores.
    """
    x = df[numeric_col]
    y = df[target_col]
    valid_idx = x.notna() & y.notna()  # Drop missing values

    try:
        auc = roc_auc_score(y[valid_idx], x[valid_idx])
        gini_abs = abs(2 * auc - 1)  # Convert AUC to Gini coefficient
        result = {'variable': numeric_col, 'auc_score': auc, 'gini_abs': gini_abs}
    except ValueError:
        result = {'variable': numeric_col, 'auc_score': None, 'gini_abs': None}
    
    return result

In [19]:
def calculate_numeric_selection_metrics(df, numeric_cols, target_col):
    """
    Calculate AUC and t-test for a numeric feature against a binary target.
    
    Args:
        df (pd.DataFrame): DataFrame containing the feature and target
        numeric_col (str): Name of the numeric feature column
        target_col (str): Name of the binary target column
    
    Returns:
        dict: AUC score and t-test results
    """

    results = []
    for col in numeric_cols:
        auc_result = calculate_auc_one_var(df, col, target_col)
        ttest_result = ttest_one_var(df, col, target_col, equal_var=False)
        missing_perc = df[col].isna().sum()/df.shape[0] * 100
        results.append({
            'variable': col,
            'missing_count': missing_perc,
            'auc_score': auc_result['auc_score'],
            'gini_abs': auc_result['gini_abs'],
            't_statistic': ttest_result['t_statistic'],
            'p_value': ttest_result['p_value'],
            'mean_group1': ttest_result['mean_group1'],
            'mean_group2': ttest_result['mean_group2'],
            'cohens_d': ttest_result['cohens_d']
        })
    
    return pd.DataFrame(results)
    

In [20]:
def logistic_regression_coef(df, feature_cols, target_col):
    """
    Fit a logistic regression model and return coefficients for each feature.
    
    Args:
        df (pd.DataFrame): DataFrame containing features and target
        feature_cols (list): List of feature column names
        target_col (str): Name of the target column
    
    Returns:
        pd.DataFrame: DataFrame with feature names and their coefficients
    """
    X = df[feature_cols]
    y = df[target_col]

    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X)

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # Fit logistic regression model
    model = LogisticRegression(random_state=123, max_iter=10000, penalty='l1', solver='liblinear', C=0.1)
    model.fit(X_scaled, y)

    # Create DataFrame with coefficients
    coef_df = pd.DataFrame({'variable': feature_cols, 'coef': model.coef_[0]})
    
    return coef_df

In [21]:
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit')

In [22]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [23]:
df = pd.read_parquet('src\\intermediate\\creditcard_processed.parquet')

In [24]:
df_with_target = df#base.merge(df, on='SK_ID_CURR', how='left')

In [25]:
numcols = df_with_target.select_dtypes(include=np.number).columns.to_list()
numcols.remove('TARGET')

In [26]:
numcols

['CREDITCARD_MAX_DURATION',
 'CREDITCARD_AVG_LIMIT_AMOUNT',
 'CREDITCARD_AVG_BALANCE_AMOUNT',
 'CREDITCARD_AVG_DRAWING_AMOUNT',
 'CREDITCARD_AVG_PAYMENT_AMOUNT',
 'CREDITCARD_MAX_UTILIZATION_RATE',
 'CREDITCARD_AVG_PAYMENT_RATE',
 'CREDITCARD_AVG_DRAWING_TO_PAYMENT_RATE',
 'CREDITCARD_AVG_INTCOVERAGE_RATE',
 'CREDITCARD_AVG_POSAMT_RATE',
 'CREDITCARD_AVG_ATMAMT_RATE',
 'CREDITCARD_AVG_POSCOUNT_RATE',
 'CREDITCARD_AVG_ATMCOUNT_RATE',
 'CREDITCARD_AVG_REPAYMENTCOUNT_RATE',
 'CREDITCARD_ISNULL_DRAWINGS_CURRENT_FLAG',
 'CREDITCARD_MAX_DPD',
 'CREDITCARD_MAX_DPDUTP',
 'CREDITCARD_DAYS_IN_DPD_RATIO',
 'CREDITCARD_AVG_AMT_BALANCE_YOY_PCT_CHANGE',
 'CREDITCARD_AVG_AMT_CREDIT_LIMIT_ACTUAL_YOY_PCT_CHANGE',
 'CREDITCARD_AVG_AMT_DRAWINGS_CURRENT_YOY_PCT_CHANGE',
 'CREDITCARD_AVG_AMT_PAYMENT_TOTAL_CURRENT_YOY_PCT_CHANGE',
 'CREDITCARD_AVG_CNT_DRAWINGS_CURRENT_YOY_ABS_CHANGE',
 'CREDITCARD_AVG_CNT_INSTALMENT_MATURE_CUM_YOY_ABS_CHANGE',
 'CREDITCARD_AVG_SK_DPD_YOY_ABS_CHANGE',
 'CREDITCARD_AVG_LY_LIM

In [27]:
numeric_selection_metrics = calculate_numeric_selection_metrics(df_with_target, numcols, 'TARGET')

C:\Users\domjo\AppData\Local\Temp\ipykernel_29140\46911737.py:30: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std
C:\Users\domjo\AppData\Local\Temp\ipykernel_29140\46911737.py:30: RuntimeWarning: invalid value encountered in scalar divide
  cohens_d = (group1.mean() - group2.mean()) / pooled_std


In [28]:
coef_df = logistic_regression_coef(df_with_target, numcols, 'TARGET')

In [30]:
numeric_selection_metrics_full = numeric_selection_metrics.merge(coef_df,
                                                                how='left', 
                                                                right_on='variable',
                                                                left_on='variable'
                                                                )

In [31]:
numeric_selection_metrics_full

,variable,missing_count,auc_score,gini_abs,t_statistic,p_value,mean_group1,mean_group2,cohens_d,coef
0,CREDITCARD_MAX_DURATION,0.0,0.560548,0.121095,-19.810753,1.283516e-85,-38.043074,-30.746113,-0.217749,0.297357
1,CREDITCARD_AVG_LIMIT_AMOUNT,0.0,0.497985,0.004031,2.867121,4.151534e-03,208687.581676,202453.112969,0.032576,-0.225033
2,CREDITCARD_AVG_BALANCE_AMOUNT,0.0,0.604635,0.209270,-22.602005,5.992363e-110,68419.446327,102137.371500,-0.310552,0.264114
3,CREDITCARD_AVG_DRAWING_AMOUNT,0.0,0.606446,0.212892,-17.454326,3.875388e-67,13463.268064,18975.896793,-0.206438,-0.256488
4,CREDITCARD_AVG_PAYMENT_AMOUNT,0.0,0.569223,0.138446,-7.433230,1.150115e-13,10285.528545,12015.265311,-0.079161,0.018762
...,...,...,...,...,...,...,...,...,...,...
105,CREDITCARD_LAST_MONTH_ISNULL_DRAWINGS_CURRENT_...,0.0,0.500000,0.000000,NaN,NaN,0.000000,0.000000,NaN,0.000000
106,CREDITCARD_LAST_MONTH_DPD_GE_1,0.0,0.502824,0.005648,-3.541313,4.002422e-04,0.012797,0.018445,-0.049333,0.012020
107,CREDITCARD_LAST_MONTH_DPD_GE_7,0.0,0.501551,0.003102,-2.254529,2.418758e-02,0.010468,0.013570,-0.030103,-0.017589
108,CREDITCARD_LAST_MONTH_DPD_GE_30,0.0,0.499322,0.001357,1.344957,1.786716e-01,0.008339,0.006983,0.015022,-0.068444


In [32]:
numeric_selection_metrics_full['points'] = ( numeric_selection_metrics_full['coef'] > 0 )* 1 +\
                                           ( numeric_selection_metrics_full['gini_abs'] > 0.1 )* 1 +\
                                           ( numeric_selection_metrics_full['p_value'] < 0.01 )* 1  

In [33]:
numeric_selection_metrics_full.to_excel('src\\Variable-Selection\\creditcard_selection_metrics_full.xlsx', index=False)